In [1]:
from wiki_dump_reader import Cleaner, iterate
from tqdm import tqdm
import re
import os
import psutil
from tabulate import tabulate
from collections import Counter
import numpy as np
from joblib import Parallel, delayed


In [2]:
language = "english"

dump_file = f"/mnt/Velocity_Vault/Wiki_Dump/Dump/{language}_dump.xml"
raw_corpus_file = f"/mnt/Velocity_Vault/Wiki_Dump/Corpus/{language}_corpus.txt"
corpus_file = f"/mnt/Velocity_Vault/Wiki_Dump/Corpus/{language}_processed.txt"

co_occur_matrix_file = f"/mnt/Velocity_Vault/Wiki_Dump/Memory/{language}_matrix.npy"

pearson_matrix_file = f"/mnt/Velocity_Vault/Wiki_Dump/Memory/{language}_pearson_matrix.npy"

svd_matrix_U_file = f"/mnt/Velocity_Vault/Wiki_Dump/Memory/{language}_svd_matrix_U.npy"
svd_matrix_S_file = f"/mnt/Velocity_Vault/Wiki_Dump/Memory/{language}_svd_matrix_S.npy"
svd_matrix_V_file = f"/mnt/Velocity_Vault/Wiki_Dump/Memory/{language}_svd_matrix_V.npy"

svd_save_files=[svd_matrix_U_file,svd_matrix_S_file,svd_matrix_V_file]

In [3]:


def write_corpus(dump, corpus):
    '''
    Processes Wikipedia XML dump and writes cleaned text to corpus file.
    Extracts titles and text, cleans HTML/formatting, removes links,
    and saves with progress tracking using tqdm.
    '''

    cleaner = Cleaner()

    total_pages = sum(1 for _ in iterate(dump))

    with open(corpus, 'w', encoding='utf-8') as output:

        with tqdm(iterate(dump), total=total_pages, unit="page") as pbar:
            for title, text in pbar:

                text = cleaner.clean_text(text)
                cleaned_text, _ = cleaner.build_links(text)
                output.write(title + '\n' + cleaned_text + '\n') # type: ignore

    print(f"Corpus saved to: {corpus}")

    return total_pages

# page_count = write_corpus(dump_file,raw_corpus_file)

In [4]:

def clean_line(line):
    
    '''Cleans English text by converting to lowercase and removing non-alphabetic characters.'''

    line = line.lower()
    cleaned = re.sub(r'[^a-z ]', ' ', line)
    cleaned = re.sub(r' +', ' ', cleaned)
    cleaned = cleaned.strip()

    return cleaned

In [5]:

def preprocess_corpus(source_file, dest_file):
    
    '''
    Preprocesses corpus file by applying language-specific cleaning.
    Reads source file, cleans each line, removes empty lines,
    and writes continuous text to destination file.
    '''

    total_lines=0

    with open(source_file, 'r', encoding='utf-8') as src:
        total_lines=0
        for _ in src:
            total_lines+=1

    try:
        with open(source_file, 'r', encoding='utf-8') as src:
            with open(dest_file, 'w', encoding='utf-8') as dest:

                first_line = True

                with tqdm(src, total=total_lines, unit="line") as pbar:
                    for line in pbar:

                        cleaned_line = clean_line(line)+" "

                        if not cleaned_line:
                            continue

                        if not first_line:
                            dest.write(' ')
                        else:
                            first_line = False

                        dest.write(cleaned_line)

    except Exception as e:
        print(f"An error occurred: {e}")

    print(f"Processed Corpus saved to: {dest_file}")
    
# preprocess_corpus(raw_corpus_file,corpus_file)

In [6]:


def display_system_info():
    
    """Display system information using tabulate"""
    
    info = [
        ["CPU Cores", os.cpu_count()],
        ["CPU Speed (GHz)", f"{(psutil.cpu_freq().max)/1000:.2f}"],
        ["Total Memory (GB)", f"{psutil.virtual_memory().total / 1e9:.2f}"],
        ["Available Memory (GB)", f"{psutil.virtual_memory().available / 1e9:.2f}"]
    ]
    print(tabulate(info, headers=["Metric", "Value"], tablefmt="grid"))




In [7]:


def get_vocabulary(filename, min_freq):
    
    """
    Reads corpus file, counts word occurrences.
    Returns a list of words sorted by frequency (highest first).
    """
    
    with open(filename, 'r') as file:
        words = file.read().strip().split()
        
    freq_dict = Counter(words)
    
    word_freq_pairs = [(word, freq) for word, freq in freq_dict.items() if freq >= min_freq]
    
    sorted_word_freq_pairs = sorted(word_freq_pairs, key=lambda x: x[1], reverse=True)
    
    vocab_list = [word for word, _ in sorted_word_freq_pairs]
    
    return vocab_list

vocabulary = get_vocabulary(corpus_file,250)



In [8]:


def create_cooccurrence_matrix(corpus_file, matrix_file, window_size, vocab):
    
    '''
    Builds a co-occurrence matrix from a text corpus using a sliding window.  
    Uses parallel processing to on rows tospeed up row computation across vocabulary words.  
    Saves the resulting matrix as a NumPy file for later use.  
    '''
    
    with open(corpus_file, 'r') as f:
        all_words = f.read().strip().split()
        
    vocab_set = set(vocab)

    for i,word in enumerate(all_words):
        if word not in vocab_set:
            all_words[i] = ""
    
    word_pos = {}
    
    for idx in tqdm(range(len(all_words)),desc="Word Position Generation"):
        word = all_words[idx]
        if word != "":
            if word not in word_pos:
                word_pos[word] = []
            word_pos[word].append(idx)
    
    k = len(vocab)
    vocab_idx = {word: i for i, word in enumerate(vocab)}
    
    def compute_row(word, all_words, vocab_idx, window_size, k):
        row = np.zeros(k, dtype=np.int32)
        if word not in word_pos:
            return row
        
        for pos in word_pos[word]:
            start = max(0, pos - window_size)
            end = min(len(all_words), pos + window_size + 1)
            
            for j in range(start, end):
                if j != pos and all_words[j] != "":
                    context_word = all_words[j]
                    distance = abs(j - pos)
                    row[vocab_idx[context_word]] += window_size - distance + 1
        
        return row
    
    rows = Parallel(n_jobs=os.cpu_count(), require='sharedmem')(
        delayed(compute_row)(word, all_words, vocab_idx, window_size, k)
        for word in tqdm(vocab, desc="Co-occurance Matrix (Row)")
    )
    
    matrix = np.array(rows, dtype=np.int32)
    
    np.save(matrix_file, matrix)
    
    print(f"Co-occurrence matrix saved to {matrix_file}")

# create_cooccurrence_matrix(corpus_file, co_occur_matrix_file, 4, vocabulary)

In [9]:


def compute_pearson_correlation(input_matrix_file, output_matrix_file):
    
    '''
    Computes the Pearson correlation matrix from a saved co-occurrence matrix.  
    Handles low-variance rows safely to avoid division errors in normalization.  
    Saves the final correlation matrix as a NumPy file for later use.  
    '''

    print("Computing variances and std and means")
    
    X = np.load(input_matrix_file).astype(np.float32)
    n = X.shape[0]
    
    row_means = np.mean(X, axis=1, keepdims=True)
    X_centered = X - row_means
    
    variances = np.var(X_centered, axis=1, ddof=1)
    
    variance_threshold = 1e-4
    low_variance_mask = variances < variance_threshold
    
    stds = np.sqrt(variances)
    stds[low_variance_mask] = 1.0
    
    X_normalized = X_centered / stds[:, np.newaxis]
    
    del X, X_centered
    
    print("Computing Pearson correlation matrix (dot product)")
    
    corr_matrix = (X_normalized @ X_normalized.T) / (n - 1)
    
    del X_normalized
    
    corr_matrix[low_variance_mask, :] = 0.0
    corr_matrix[:, low_variance_mask] = 0.0
    np.fill_diagonal(corr_matrix, 1.0)
    
    np.save(output_matrix_file, corr_matrix)
    
    print(f"Pearson correlation matrix saved at {output_matrix_file}")

# compute_pearson_correlation(co_occur_matrix_file, pearson_matrix_file)

In [10]:
def display_similar_words(pcm_file, vocab_list, target_words):
    '''
    Display top 5 similar words with cosine similarity directly from
    the Pearson correlation matrix. Each target word is shown as a 
    column header with the top 5 most similar words listed below.
    '''
    
    P = np.load(pcm_file).astype(np.float32)
    results = []
    
    for target_word in target_words:
        target_idx = vocab_list.index(target_word)
        pearson_scores = P[target_idx, :]
        
        top_indices = np.argsort(pearson_scores)[::-1][1:6]
        
        similar_lines = [
            f"{vocab_list[idx]} ({pearson_scores[idx]:.4f})"
            for idx in top_indices
        ]
        
        results.append("\n".join(similar_lines))
    
    table_data = [results]
    print(tabulate(table_data, headers=target_words, tablefmt="grid"))


In [11]:

def compute_svd(pcm_file,save_files):
    
    '''
    Performs Singular Value Decomposition (SVD) on a given matrix file.  
    Extracts U, Σ, and Vᵀ components using NumPy’s SVD routine.  
    Saves each component to disk and returns them for further processing.  
    '''

    P = np.load(pcm_file).astype(np.float32)
    
    U, sigma, Vt = np.linalg.svd(P, full_matrices=False)
    
    np.save(save_files[0],np.array(U))
    np.save(save_files[1],np.array(sigma))
    np.save(save_files[2],np.array(Vt))

    return U, sigma, Vt

def compute_embeddings(save_files, embedding_dim):
    
    '''
    Generates word embeddings from the saved SVD decomposition.  
    Selects the top-k singular vectors and scales them by sqrt(Σ).  
    Returns a dense embedding matrix of the desired dimension.  
    '''


    U = np.load(save_files[0]).astype(np.float32)
    sigma = np.load(save_files[1]).astype(np.float32)
    
    # Vt = np.load(save_files[2]).astype(np.float32)
    
    U_k = U[:, :embedding_dim]
    
    # V_k = Vt[:embedding_dim, :].T
    # Sigma_k = np.diag(sigma[:embedding_dim])

    sqrt_sigma = np.diag(np.sqrt(sigma[:embedding_dim]))
    
    return U_k @ sqrt_sigma



In [12]:
# U, sigma, Vt = compute_svd(pearson_matrix_file, svd_save_files)

word_vectors = compute_embeddings(svd_save_files, embedding_dim=500)

In [13]:


def display_svd_similarities(vocab_list, target_words, word_vectors, top_k=5):
    
    """
    Display similarities using SVD for both finding and scoring words.
    Each target word gets its own column, with the top_k neighbors listed.
    """
    
    norms = np.linalg.norm(word_vectors, axis=1, keepdims=True)
    norms[norms == 0] = 1e-10  
    word_vectors = word_vectors / norms

    results = []
    target_indices = [vocab_list.index(word) for word in target_words]

    for target_idx in target_indices:
        target_vector = word_vectors[target_idx]

        sims = word_vectors @ target_vector

        sims[target_idx] = -np.inf

        top_indices = np.argsort(sims)[::-1][:top_k]
        similar_words = [f"{vocab_list[i]} ({sims[i]:.4f})" for i in top_indices]

        results.append("\n".join(similar_words))

    table_data = [results]
    print(tabulate(table_data, headers=target_words, tablefmt="grid"))


In [14]:
display_system_info()

print("\n\nVocabulary Size - ",len(vocabulary))

+-----------------------+---------+
| Metric                |   Value |
+=======================+=========+
| CPU Cores             |   15    |
+-----------------------+---------+
| CPU Speed (GHz)       |    3.94 |
+-----------------------+---------+
| Total Memory (GB)     |   24.87 |
+-----------------------+---------+
| Available Memory (GB) |   18.62 |
+-----------------------+---------+


Vocabulary Size -  22590


In [18]:
target_words = vocabulary[720:725]

In [19]:
display_similar_words(pearson_matrix_file, vocabulary, target_words)

+-------------------+-------------------+--------------------+------------------------+--------------------+
| plan              | front             | able               | management             | republic           |
+===================+===================+====================+========================+====================+
| team (0.8703)     | fall (0.9815)     | obliged (0.9694)   | development (0.9711)   | emirate (0.9710)   |
| lottery (0.8680)  | context (0.9776)  | unable (0.9685)    | wisdom (0.9651)        | creation (0.9710)  |
| anthem (0.8648)   | spring (0.9771)   | subjected (0.9679) | documentation (0.9644) | emergence (0.9701) |
| monument (0.8647) | presence (0.9754) | willing (0.9648)   | regeneration (0.9634)  | battle (0.9680)    |
| fund (0.8639)     | history (0.9742)  | unwilling (0.9538) | discipline (0.9616)    | invasion (0.9675)  |
+-------------------+-------------------+--------------------+------------------------+--------------------+


In [20]:
display_svd_similarities(vocabulary, target_words, word_vectors)

+------------------+------------------+--------------------+------------------------+----------------------+
| plan             | front            | able               | management             | republic             |
+==================+==================+====================+========================+======================+
| sixth (0.9238)   | fall (0.9857)    | obliged (0.9831)   | regeneration (0.9800)  | emirate (0.9918)     |
| lottery (0.9235) | chapter (0.9844) | subjected (0.9803) | development (0.9796)   | conquest (0.9889)    |
| guard (0.9225)   | spring (0.9839)  | unable (0.9794)    | documentation (0.9769) | invasion (0.9883)    |
| fund (0.9082)    | context (0.9829) | willing (0.9780)   | governance (0.9769)    | creation (0.9876)    |
| team (0.9055)    | murders (0.9805) | unwilling (0.9689) | diversity (0.9769)     | suppression (0.9875) |
+------------------+------------------+--------------------+------------------------+----------------------+
